# 01 · Arrays & Hashing

> Set e dizionari per sapere in `O(1)` se hai già visto qualcosa, contare e raggruppare. È la base di quasi tutte le altre categorie.
> Idee per i singoli problemi: [Pattern algoritmici › Hashing](../Fundamentals/Pattern%20algoritmici.md#hashing-e-conteggi) · Costi delle operazioni: [Big-O › Hash map e set](../Fundamentals/Big-O%20e%20strutture%20dati.md#hash-map-e-set)

**Come usarlo:** cerca il problema nella tabella, poi salta alla sezione che spiega la sintassi che ti manca. Il notebook non contiene le soluzioni.

## Problemi → cosa ti serve

| Problema | Concetti | Sintassi |
|---|---|---|
| Contains Duplicate | set come "già visto" | `set()`, `x in seen`, `len(set(nums))` |
| Valid Anagram | stesse frequenze | `Counter(s) == Counter(t)`, `sorted(s)` |
| Two Sum | complemento già visto → indice | `dict`, `enumerate`, `target - x in d` |
| Group Anagrams | stessa chiave per parole equivalenti | `defaultdict(list)`, chiave `tuple` (hashable) |
| Top K Frequent Elements | conta, poi ordina per frequenza (o bucket) | `Counter.most_common(k)`, `[[] for _ in range(n + 1)]` |
| Encode and Decode Strings | prefisso di lunghezza: `"5#hello"` | `str.find("#", i)`, slicing `s[i:j]`, `int()` |
| Product of Array Except Self | prodotti prefissi × suffissi | `[1] * n`, `range(n - 1, -1, -1)` |
| Valid Sudoku | un set per riga, colonna e box | `defaultdict(set)`, chiave `(r // 3, c // 3)` |
| Longest Consecutive Sequence | parti solo dagli inizi di sequenza | `set(nums)`, `x - 1 not in s` |

## Sintassi al volo

| Operazione | `set` | `dict` |
|---|---|---|
| crea | `set()`, `{1, 2}`, `set(xs)` | `{}`, `dict(zip(keys, vals))`, `{k: v for ...}` |
| aggiungi | `s.add(x)` | `d[k] = v` |
| esiste? | `x in s` | `k in d` (cerca tra le **chiavi**) |
| leggi con default | — | `d.get(k, 0)` |
| rimuovi | `s.discard(x)` (nessun errore), `s.remove(x)` | `d.pop(k, None)`, `del d[k]` |
| itera | `for x in s` | `for k, v in d.items()` |
| operazioni | `a \| b`, `a & b`, `a - b` | `d1 \| d2` (merge, 3.9+) |

Tutte le operazioni sopra sono `O(1)` medio. Una **chiave** deve essere *hashable* (immutabile): `int`, `str`, `tuple` sì; `list`, `set`, `dict` no.

## Set: "l'ho già visto?"

Un set risponde a `x in seen` in tempo costante, mentre su una lista costa `O(n)`. Due modi tipici: confronto di lunghezze oppure controllo durante il ciclo, che si ferma al primo duplicato.

In [1]:
skus = ["TSHIRT-M", "MUG-01", "CAP-RED", "MUG-01"]

print(len(set(skus)) != len(skus))  # -> True: c'è almeno un duplicato

seen = set()
for sku in skus:
    if sku in seen:
        print("duplicato:", sku)  # -> duplicato: MUG-01
        break
    seen.add(sku)

True
duplicato: MUG-01


## Dict: valore → informazione (indice, conteggio, lista)

Il caso classico è memorizzare *dove* hai visto un valore. `enumerate` ti dà indice e valore insieme; `d.get(k, default)` evita il `KeyError`.

In [2]:
prices = [1200, 990, 2599, 450]
budget = 1440

index_of: dict[int, int] = {}
for i, price in enumerate(prices):
    missing = budget - price
    if missing in index_of:
        print(index_of[missing], i)  # -> 1 3 (990 + 450)
    index_of[price] = i              # inserisci DOPO il controllo: non accoppi un elemento con sé stesso

1 3


## Contare: `Counter` e `dict.get`

`Counter` è un `dict` specializzato: le chiavi mancanti valgono 0, due `Counter` si confrontano con `==`, `most_common(k)` restituisce le `k` coppie più frequenti.

In [3]:
from collections import Counter

order_skus = ["MUG", "CAP", "MUG", "TSHIRT", "MUG", "CAP"]
counts = Counter(order_skus)
print(counts["MUG"], counts["SOCKS"])        # -> 3 0 (nessun KeyError)
print(counts.most_common(2))                 # -> [('MUG', 3), ('CAP', 2)]
print(Counter("listen") == Counter("silent"))  # -> True

# stessa cosa a mano, con un dict normale
manual: dict[str, int] = {}
for sku in order_skus:
    manual[sku] = manual.get(sku, 0) + 1
print(manual)  # -> {'MUG': 3, 'CAP': 2, 'TSHIRT': 1}

3 0
[('MUG', 3), ('CAP', 2)]
True
{'MUG': 3, 'CAP': 2, 'TSHIRT': 1}


**Conteggio in un array di 26:** se l'input è fatto solo di lettere minuscole, `ord(ch) - ord("a")` dà un indice 0–25. L'array va convertito in `tuple` per usarlo come chiave di un dict.

In [4]:
def letter_counts(word: str) -> tuple[int, ...]:
    counts = [0] * 26
    for ch in word:
        counts[ord(ch) - ord("a")] += 1
    return tuple(counts)  # tuple: hashable, può fare da chiave

print(letter_counts("eat") == letter_counts("tea"))  # -> True
print(chr(ord("a") + 2))                             # -> c (operazione inversa)

True
c


## Raggruppare: `defaultdict` e chiavi composte

`defaultdict(list)` crea la lista vuota al primo accesso a una chiave nuova: niente `if key not in d`. La chiave del gruppo deve rappresentare la proprietà comune (qui: la categoria; per gli anagrammi, una "firma" della parola).

In [5]:
from collections import defaultdict

products = [("MUG", "kitchen"), ("CAP", "apparel"), ("PLATE", "kitchen")]

by_category: defaultdict[str, list[str]] = defaultdict(list)
for sku, category in products:
    by_category[category].append(sku)
print(dict(by_category))         # -> {'kitchen': ['MUG', 'PLATE'], 'apparel': ['CAP']}
print(list(by_category.values()))  # -> [['MUG', 'PLATE'], ['CAP']]

print(tuple(sorted("bca")), "".join(sorted("bca")))  # -> ('a', 'b', 'c') abc: due firme valide

{'kitchen': ['MUG', 'PLATE'], 'apparel': ['CAP']}
[['MUG', 'PLATE'], ['CAP']]
('a', 'b', 'c') abc


⚠️ `d[["a", "b"]] = 1` solleva `TypeError: unhashable type: 'list'`. Converti con `tuple(...)` o `"".join(...)`.

`defaultdict(set)` e `defaultdict(int)` funzionano allo stesso modo: utili per *Valid Sudoku* (un set per ogni riga/colonna/box).

## Ordinare e top-k

`sorted(xs, key=..., reverse=True)` restituisce una nuova lista; `xs.sort()` ordina sul posto e restituisce `None`. Costo `O(n log n)`.

**Bucket sort** per le frequenze: la frequenza di un elemento sta tra 1 e `n`, quindi puoi usare una lista di liste indicizzata dalla frequenza e scorrerla dal fondo, in `O(n)`.

In [6]:
from collections import Counter

sales = ["MUG", "CAP", "MUG", "TSHIRT", "MUG", "CAP"]
counts = Counter(sales)

by_freq = sorted(counts, key=lambda sku: counts[sku], reverse=True)
print(by_freq)  # -> ['MUG', 'CAP', 'TSHIRT']

buckets: list[list[str]] = [[] for _ in range(len(sales) + 1)]  # NON [[]] * n
for sku, freq in counts.items():
    buckets[freq].append(sku)
print(buckets)  # -> [[], ['TSHIRT'], ['CAP'], ['MUG'], [], [], []]

for freq in range(len(buckets) - 1, 0, -1):  # dal più frequente
    if buckets[freq]:
        print(freq, buckets[freq])  # -> 3 ['MUG'] (poi 2, poi 1)

['MUG', 'CAP', 'TSHIRT']
[[], ['TSHIRT'], ['CAP'], ['MUG'], [], [], []]
3 ['MUG']
2 ['CAP']
1 ['TSHIRT']


⚠️ `[[]] * n` crea `n` riferimenti **alla stessa lista**: un `append` compare ovunque. Usa sempre la comprehension.

## Stringhe: slicing, `find`, `join`

| Serve | Sintassi |
|---|---|
| sottostringa | `s[i:j]` (j escluso), `s[i:]`, `s[-3:]` |
| posizione di un carattere da un indice in poi | `s.find("#", i)` → `-1` se manca; `s.index` solleva `ValueError` |
| unire pezzi | `"".join(parts)`, `",".join(map(str, nums))` |
| dividere | `s.split(",")` |
| numero ↔ stringa | `int("42")`, `str(42)`, `f"{len(w)}#{w}"` |

Costruire una stringa con `+=` in un ciclo crea una copia a ogni passo: raccogli i pezzi in una lista e usa `join`.

In [7]:
# Prefisso di lunghezza: funziona anche se le parole contengono il separatore.
names = ["Ada", "#lin", ""]
encoded = "".join(f"{len(name)}#{name}" for name in names)
print(encoded)  # -> 3#Ada4##lin0#

i = 0
hash_pos = encoded.find("#", i)
length = int(encoded[i:hash_pos])
print(length, encoded[hash_pos + 1 : hash_pos + 1 + length])  # -> 3 Ada

3#Ada4##lin0#
3 Ada


## Prefissi e suffissi

Un **prefix** in posizione `i` riassume tutto ciò che sta *prima* di `i` (somma, prodotto, massimo); un **suffix**, tutto ciò che sta *dopo*. Con due passate (avanti e indietro) ottieni per ogni posizione l'informazione "tutto tranne me" senza ricalcolarla.

In [8]:
daily_orders = [3, 1, 4, 2]
n = len(daily_orders)

prefix = [0] * n  # prefix[i] = somma degli elementi prima di i
for i in range(1, n):
    prefix[i] = prefix[i - 1] + daily_orders[i - 1]

suffix = [0] * n  # suffix[i] = somma degli elementi dopo i
for i in range(n - 2, -1, -1):  # range all'indietro: start, stop escluso, passo -1
    suffix[i] = suffix[i + 1] + daily_orders[i + 1]

print(prefix, suffix)                         # -> [0, 3, 4, 8] [7, 6, 2, 0]
print([p + s for p, s in zip(prefix, suffix)])  # -> [7, 9, 6, 8]: somma di tutti tranne i

[0, 3, 4, 8] [7, 6, 2, 0]
[7, 9, 6, 8]


Con `range(n - 1, -1, -1)` scorri gli indici da `n - 1` a `0`; in alternativa `reversed(range(n))`. La memoria extra si può eliminare accumulando il suffisso in una variabile mentre scrivi nel risultato.

## Griglie: indice del box 3×3

In una griglia 9×9, la divisione intera `//` raggruppa righe e colonne a blocchi di 3: `(r // 3, c // 3)` identifica il box (da `(0, 0)` a `(2, 2)`).

In [9]:
from collections import defaultdict

box_cells: defaultdict[tuple[int, int], list] = defaultdict(list)
for r in range(9):
    for c in range(9):
        box_cells[(r // 3, c // 3)].append((r, c))

print(len(box_cells), box_cells[(1, 2)][:3])  # -> 9 [(3, 6), (3, 7), (3, 8)]
r, c = 4, 7
print((r // 3) * 3 + c // 3)  # -> 5: alternativa, box numerato da 0 a 8

9 [(3, 6), (3, 7), (3, 8)]
5


## Set per sequenze e "inizi"

Un trucco ricorrente: con tutti i valori in un set, un valore `x` è l'**inizio** di una sequenza consecutiva se `x - 1` non c'è. Partendo solo dagli inizi, ogni elemento viene visitato al più due volte: `O(n)` totale anche con un `while` annidato.

In [10]:
order_ids = {101, 4, 102, 2, 3, 200}

starts = [x for x in order_ids if x - 1 not in order_ids]
print(sorted(starts))  # -> [2, 101, 200]

[2, 101, 200]


## Errori comuni

- Modificare un dict o set mentre lo iteri: `RuntimeError`. Itera su una copia (`list(d)`).
- `sorted(d)` ordina le **chiavi**; per le coppie usa `sorted(d.items(), key=lambda kv: kv[1])`.
- `dict` mantiene l'ordine di inserimento (3.7+), `set` no: non affidarti all'ordine di un set.
- Two Sum: inserisci nel dict **dopo** aver cercato il complemento.